In [1]:
!pip install "transformers>=4.37.0" accelerate

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [3]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

print("Model loaded successfully!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!


In [4]:
def generate_response(messages):
    # Converting the chat messages into a single formatted prompt
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Converting the text into tokens
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    # Generating the response using the transformer model
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )

    # Removing the input tokens and keep only newly generated tokens
    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    # Converting the generated tokens back to readable text
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    # Returning the final chatbot response
    return response

In [5]:
def chatbot():
    """
    Console-based chatbot function
    Call chatbot() to start interaction
    """

    # Initial default message
    print("Chatbot: Hello! I am your AI assistant. Type 'exit' or 'quit' to stop.")

    # Initializing conversation history
    messages = [
        {"role": "system", "content": "You are a helpful and friendly AI assistant."}
    ]

    while True:
        # Taking the user input
        user_input = input("You: ")

        # applyong the exit condition
        if user_input.lower() in ["exit", "quit"]:
            print("Chatbot: Goodbye! Have a great day.")
            break

        # Adding user message to history
        messages.append({
            "role": "user",
            "content": user_input
        })

        # Generating the response from the model
        response = generate_response(messages)

        # Printing the chatbot response
        print("Chatbot:", response)

        # Adding all the bot response to history
        messages.append({
            "role": "assistant",
            "content": response
        })

In [6]:
chatbot()

Chatbot: Hello! I am your AI assistant. Type 'exit' or 'quit' to stop.
You: hi whats the time ?
Chatbot: Hello! I'm here to help you with any questions or concerns you may have. What would you like to know about time?
You: whats the time now?
Chatbot: As an artificial intelligence, I don't have access to real-time information. However, you can check the current time on your device by looking at the clock or using a navigation app if you're traveling. If you need more specific information or assistance related to time-related tasks, feel free to ask!
You: what is AI?
Chatbot: AI stands for Artificial Intelligence, which refers to the simulation of human intelligence in machines that are designed to think and learn like humans. It involves the development of algorithms, models, and systems that enable machines to perform tasks that typically require human intelligence, such as visual perception, speech recognition, decision-making, problem-solving, and natural language processing.

Artif